In [1]:
!pip install ollama pydantic python-dotenv pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.9/444.9 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 57.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 365.9 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0


In [3]:
# -*- coding: utf-8 -*-
"""
Procesador de textos con salida estructurada (Ollama local).
- Usa Ollama en http://localhost:11434 (ajústalo con OLLAMA_HOST si es distinto).
- Fuerza salida JSON válida con JSON Schema (structured outputs).
- Valida y guarda en JSONL y CSV.

Requisitos:
    pip install ollama pydantic python-dotenv pandas
"""

import os
import json
import csv
import time
from typing import Dict, List, Any, Optional
import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from ollama import Client

# =========================
# Configuración
# =========================
load_dotenv()
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://ollama:11434")
MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b-instruct-fp16")
client = Client(host=OLLAMA_HOST)

# =========================
# Datos de entrada
# =========================
ITEMS: List[Dict[str, str]] = [
    {
        "item_key": "item1",
        "texto": (
            "Zanja para tendido de cables de 1,10 m de profundidad y 0,60 m de anchura máxima, "
            "zona de balasto, ejecutada a mano incluso descerne y entibación de costados y posterior "
            "reposición del balasto retirado, y relleno compactado con material seleccionado procedente "
            "de la excavación, cama de arena de 40 cm de espesor, rejilla de plástico identificadora, "
            "carga, descarga y transporte al lugar asignado en el interior de la obra.\n"
            "Trabajo: Cualquier franja horaria\n"
            "Banda de mantenimiento: No necesita intervalo\n"
            "Condiciones de ejecución: Cualquier condición de ejecución"
        )
    },
    {
        "item_key": "item2",
        "texto": (
            "Zanja para tendido de cables de hasta 0,80 m de profundidad y 0,60 m de anchura máxima, "
            "ejecutada por excavación manual en cualquier tipo de terreno excepto roca, y relleno compactado "
            "con material seleccionado procedente de la excavación, cama de arena de 20 cm de espesor, "
            "rejilla de plastico identificadora, carga, descarga y transporte al lugar asignado en el interior "
            "de la obra.\n"
            "Trabajo: Diurno\n"
            "Banda de mantenimiento: i >= 5 horas\n"
            "Condiciones de ejecución: Volumen relevante\n"
        )
    },
    {
        "item_key": "item3",
        "texto": (
            "Canalización hormigonada de 1 tubos de polietileno libre de halógenos "
            "de 110 mm de diámetro en cualquier clase de terreno, excepto roca, "
            "incluso el relleno y compactado de la zanja, el suministro y montaje "
            "de los tubos y hormigón tipo HE-20 sin vibrar, la prueba de los conductos, "
            "el transporte y la retirada de los productos al lugar de empleo.\n"
            "Trabajo: Diurno\n"
            "Banda de mantenimiento: i >= 5 horas\n"
            "Condiciones de ejecución: Volumen relevante\n"
        )
    }
]

# =========================
# Parámetros de validación
# =========================
MAX_PALABRAS = 500
MIN_CARACTERES = 3

# =========================
# Prompt base
# =========================
PROMPT_TEMPLATE = (
    "Transforma el texto siguiendo estrictamente estas pautas:\n"
    "1) Mantén el significado original y todas las palabras clave.\n"
    "2) Desarrolla el texto añadiendo detalles constructivos adicionales sin alterar la naturaleza ni el dimensionamiento.\n"
    "3) No incluyas explicaciones ni comentarios, solo el texto final.\n"
    "Devuelve ÚNICAMENTE un JSON que contenga una sola clave: 'texto_alterado'.\n"
    "Máximo {max_palabras} palabras.\n\n"
    "Texto a transformar:\n{texto}"
)

def build_prompt(texto: str, max_palabras: int = MAX_PALABRAS) -> str:
    return PROMPT_TEMPLATE.format(texto=texto, max_palabras=max_palabras)

# =========================
# JSON Schema (solo texto_alterado)
# =========================
class Salida(BaseModel):
    texto_alterado: str = Field(..., min_length=MIN_CARACTERES)

SALIDA_SCHEMA: Dict[str, Any] = Salida.model_json_schema()

# =========================
# Utilidades
# =========================
def parse_json_strict(s: str) -> Dict[str, Any]:
    """Intenta cargar JSON incluso si vienen backticks u otros adornos."""
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        s2 = s.strip().strip("```").strip()
        return json.loads(s2)

def llamar_ollama(prompt: str,
                  model: str = MODEL,
                  max_reintentos: int = 3,
                  pausa_base: float = 1.0) -> Dict[str, Any]:
    intento = 0
    while True:
        try:
            resp = client.chat(
                model=model,
                messages=[
                    {"role": "system", "content": "Eres un asistente que responde SOLO en JSON válido."},
                    {"role": "user", "content": prompt}
                ],
                format=SALIDA_SCHEMA,
                options={"temperature": 0, "num_ctx": 2048}
            )
            return parse_json_strict(resp["message"]["content"])
        except Exception as e:
            intento += 1
            if intento > max_reintentos:
                raise
            time.sleep(pausa_base * (2 ** (intento - 1)))

def validar_texto(texto_alterado: str) -> Optional[str]:
    if not texto_alterado or not texto_alterado.strip():
        return "texto_alterado vacío."
    if len(texto_alterado.strip()) < MIN_CARACTERES:
        return f"texto_alterado demasiado corto (<{MIN_CARACTERES} caracteres)."
    num_palabras = len(texto_alterado.split())
    if num_palabras > MAX_PALABRAS:
        return f"Excede el máximo de {MAX_PALABRAS} palabras (tiene {num_palabras})."
    return None

# =========================
# Proceso principal
# =========================
def procesar_items(items: List[Dict[str, str]]) -> List[Dict[str, str]]:
    resultados: List[Dict[str, str]] = []
    errores: List[Dict[str, str]] = []

    for item in items:
        prompt = build_prompt(item["texto"])
        try:
            data = llamar_ollama(prompt)
            texto_alterado = data.get("texto_alterado", "").strip()
            motivo = validar_texto(texto_alterado)
            if motivo:
                errores.append({"item_key": item["item_key"], "motivo": motivo})
                print(f"⚠️ {item['item_key']}: {motivo}")
                continue
            # Añadimos nosotros el item_key
            resultados.append({"item_key": item["item_key"], "texto_alterado": texto_alterado})
        except Exception as e:
            errores.append({"item_key": item["item_key"], "motivo": f"Excepción: {e}"})
            print(f"❌ {item['item_key']}: {e}")

    # Persistencia
    if resultados:
        with open("resultados.jsonl", "w", encoding="utf-8") as f:
            for r in resultados:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        with open("resultados.csv", "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["item_key", "texto_alterado"])
            writer.writeheader()
            writer.writerows(resultados)

    # Reporte
    print("\n=== RESUMEN ===")
    print(f"Procesados: {len(items)} | OK: {len(resultados)} | Errores: {len(errores)}")
    for e in errores:
        print(f"- {e['item_key']}: {e['motivo']}")

    return resultados

In [4]:
resultados = process_items(ITEMS)

Total: 3 | Already done: 0 | Pending: 3
Concurrency: 2 | Model: llama3.1:8b-instruct-fp16 | num_ctx=768 | num_predict=700


Processing: 100%|██████████| 3/3 [20:33<00:00, 411.11s/it]


=== SUMMARY ===
Processed: 3 | OK: 0 | Errors: 3
- item2: Exception: HTTPConnectionPool(host='ollama', port=11434): Read timed out. (read timeout=120)
- item1: Exception: HTTPConnectionPool(host='ollama', port=11434): Read timed out. (read timeout=120)
- item3: Exception: HTTPConnectionPool(host='ollama', port=11434): Read timed out. (read timeout=120)


In [11]:
resultados

[{'item_key': 'item1',
  'texto_alterado': 'Se realizará una zanja para tendido de cables con una profundidad máxima de 1,10 m y una anchura máxima de 0,60 m. La zona de balasto será excavada a mano, incluyendo la descerne y entibación de costados, y posteriormente se repondrá el balasto retirado. Posteriormente, se realizará un relleno compactado con material seleccionado procedente de la excavación. Además, se colocará una cama de arena de 40 cm de espesor y se instalará una rejilla de plástico identificadora. El transporte del material necesario se realizará dentro del interior de la obra, desde el lugar de carga hasta el lugar asignado para su descarga. Se incluye el transporte de los equipos necesarios para la ejecución del trabajo. El trabajo podrá realizarse en cualquier franja horaria y no requerirá intervalos de mantenimiento. Las condiciones de ejecución serán flexibles y se adaptarán a cualquier condición meteorológica o de seguridad que se presente durante la ejecución del 

In [ ]:
# Convertir a DataFrames
df_items = pd.DataFrame(ITEMS)
df_resultados = pd.DataFrame(resultados)

# Unir por item_key
df_final = pd.merge(df_items, df_resultados, on="item_key", how="inner")

# Visualización simple por consola
print("\n=== RESULTADOS ===")
pd.set_option("display.max_colwidth", None)   # no truncar celdas
pd.set_option("display.width", 0)            # ajuste automático del ancho
pd.set_option("display.max_columns", None)   # mostrar todas las columnas
df_final